In [61]:
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from transformers import BertTokenizer, BertModel
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score

In [62]:
train_df = pd.read_parquet("/kaggle/input/datasets/shashvatrajora/parsed/training_30.parquet")
val_df   = pd.read_parquet("/kaggle/input/datasets/shashvatrajora/parsed/validation.parquet")
test_df  = pd.read_parquet("/kaggle/input/datasets/shashvatrajora/parsed/testing.parquet")

In [63]:
STANDARD_AA = set(list("ACDEFGHIKLMNPQRSTVWY"))

def clean_residue(r):
    r = str(r).strip().upper()
    return r if r in STANDARD_AA else "X"

for df in [train_df, val_df, test_df]:
    if "Mask" not in df.columns:
        df["Mask"] = 1
    df["Mask"] = df["Mask"].fillna(0).astype(int)
    df["ResidueIndex"] = df["ResidueIndex"].astype(int)
    df["Residue"] = df["Residue"].astype(str)

In [64]:
def build_sequence_samples_from_parquet(df):
    samples = []

    for pid, g in tqdm(df.groupby("ProteinID", sort=False), desc="Building samples"):
        g = g.sort_values("ResidueIndex").reset_index(drop=True)

        seq = "".join(clean_residue(r) for r in g["Residue"].tolist())
        coords = g[["X", "Y", "Z"]].values.astype(np.float32)
        mask = g["Mask"].values.astype(np.int8)

        coord_valid = ~(np.isclose(coords, 0.0).all(axis=1))
        residue_valid = (mask == 1) & coord_valid

        L = len(g)
        targets = np.full((L, 4), np.nan, dtype=np.float32)

        for i in range(L):
            if not residue_valid[i]:
                continue
            for k, step in enumerate([1, 2, 4, 8]):
                j = i + step
                if j < L and residue_valid[j]:
                    targets[i, k] = np.linalg.norm(coords[i] - coords[j])

        valid_mask = ~np.isnan(targets).any(axis=1)

        samples.append({
            "ProteinID": pid,
            "sequence": seq,
            "targets": targets,
            "valid_mask": valid_mask.astype(np.float32)
        })
    return samples

In [65]:
train_samples = build_sequence_samples_from_parquet(train_df)
val_samples   = build_sequence_samples_from_parquet(val_df)
test_samples  = build_sequence_samples_from_parquet(test_df)

def filter_samples(samples, min_valid_residues=10):
    return [s for s in samples if int(s["valid_mask"].sum()) >= min_valid_residues]

train_samples = filter_samples(train_samples, 10)
val_samples   = filter_samples(val_samples, 10)
test_samples  = filter_samples(test_samples, 10)

print(len(train_samples), len(val_samples), len(test_samples))

Building samples:   0%|          | 0/16973 [00:00<?, ?it/s]

Building samples:   0%|          | 0/224 [00:00<?, ?it/s]

Building samples:   0%|          | 0/116 [00:00<?, ?it/s]

16821 223 116


In [66]:
MODEL_NAME = "Rostlab/prot_bert"

tokenizer = BertTokenizer.from_pretrained(MODEL_NAME, do_lower_case=False)
encoder = BertModel.from_pretrained(MODEL_NAME).to(device)

for p in encoder.parameters():
    p.requires_grad = False

encoder.eval()
print("Loaded ProtBERT")

tokenizer_config.json:   0%|          | 0.00/86.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/81.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/361 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/1.68G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/487 [00:00<?, ?it/s]

BertModel LOAD REPORT from: Rostlab/prot_bert
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.predictions.decoder.weight             | UNEXPECTED |  | 
cls.predictions.decoder.bias               | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Loaded ProtBERT


In [67]:
class ProteinDistanceDataset(Dataset):
    def __init__(self, samples, tokenizer, max_length=512):
        self.samples = samples
        self.tokenizer = tokenizer
        self.max_length = max_length

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        item = self.samples[idx]
        seq = item["sequence"]
        targets = item["targets"].copy()
        valid_mask = item["valid_mask"].copy()

        seq_spaced = " ".join(list(seq))
        enc = self.tokenizer(
            seq_spaced,
            return_tensors="pt",
            truncation=True,
            max_length=self.max_length,
            padding=False
        )

        input_ids = enc["input_ids"].squeeze(0)
        attention_mask = enc["attention_mask"].squeeze(0)

        usable_len = min(len(seq), input_ids.shape[0] - 2, len(targets), len(valid_mask))

        targets = np.nan_to_num(targets[:usable_len], nan=0.0, posinf=0.0, neginf=0.0)
        valid_mask = np.nan_to_num(valid_mask[:usable_len], nan=0.0, posinf=0.0, neginf=0.0)
        valid_mask = (valid_mask > 0).astype(np.float32)

        return {
            "input_ids": input_ids,
            "attention_mask": attention_mask,
            "targets": torch.tensor(targets, dtype=torch.float32),
            "valid_mask": torch.tensor(valid_mask, dtype=torch.float32),
            "seq_len": usable_len
        }

In [68]:
pad_token_id = tokenizer.pad_token_id

def collate_fn(batch):
    max_token_len = max(x["input_ids"].shape[0] for x in batch)
    max_res_len = max(x["seq_len"] for x in batch)

    input_ids_batch, attention_mask_batch = [], []
    targets_batch, valid_mask_batch = [], []

    for x in batch:
        input_ids = x["input_ids"]
        attention_mask = x["attention_mask"]
        targets = x["targets"]
        valid_mask = x["valid_mask"]

        tok_pad = max_token_len - input_ids.shape[0]
        if tok_pad > 0:
            input_ids = torch.cat([input_ids, torch.full((tok_pad,), pad_token_id, dtype=torch.long)])
            attention_mask = torch.cat([attention_mask, torch.zeros(tok_pad, dtype=torch.long)])

        res_pad = max_res_len - targets.shape[0]
        if res_pad > 0:
            targets = torch.cat([targets, torch.zeros((res_pad, 4), dtype=torch.float32)])
            valid_mask = torch.cat([valid_mask, torch.zeros(res_pad, dtype=torch.float32)])

        input_ids_batch.append(input_ids)
        attention_mask_batch.append(attention_mask)
        targets_batch.append(targets)
        valid_mask_batch.append(valid_mask)

    return {
        "input_ids": torch.stack(input_ids_batch),
        "attention_mask": torch.stack(attention_mask_batch),
        "targets": torch.stack(targets_batch),
        "valid_mask": torch.stack(valid_mask_batch)
    }

In [69]:
train_dataset = ProteinDistanceDataset(train_samples, tokenizer, max_length=512)
val_dataset   = ProteinDistanceDataset(val_samples, tokenizer, max_length=512)
test_dataset  = ProteinDistanceDataset(test_samples, tokenizer, max_length=512)

train_loader = DataLoader(train_dataset, batch_size=1, shuffle=True, collate_fn=collate_fn)
val_loader   = DataLoader(val_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)
test_loader  = DataLoader(test_dataset, batch_size=1, shuffle=False, collate_fn=collate_fn)

In [70]:
class ProtBERTRegressor(nn.Module):
    def __init__(self, encoder, hidden_size):
        super().__init__()
        self.encoder = encoder
        self.head = nn.Sequential(
            nn.Linear(hidden_size, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, 4)
        )

    def forward(self, input_ids, attention_mask):
        with torch.no_grad():
            out = self.encoder(input_ids=input_ids, attention_mask=attention_mask)
        hidden = out.last_hidden_state[:, 1:-1, :]
        return self.head(hidden)

In [71]:
model = ProtBERTRegressor(encoder, encoder.config.hidden_size).to(device)
optimizer = torch.optim.Adam(model.head.parameters(), lr=5e-4)

In [72]:
def masked_regression_loss(preds, targets, valid_mask):
    usable_len = min(preds.shape[1], targets.shape[1], valid_mask.shape[1])

    preds = torch.nan_to_num(preds[:, :usable_len, :], nan=0.0, posinf=0.0, neginf=0.0)
    targets = torch.nan_to_num(targets[:, :usable_len, :], nan=0.0, posinf=0.0, neginf=0.0)
    valid_mask = torch.nan_to_num(valid_mask[:, :usable_len], nan=0.0, posinf=0.0, neginf=0.0)

    mask = valid_mask.unsqueeze(-1)
    diff = torch.abs(preds - targets) * mask

    denom = mask.sum() * preds.shape[-1]
    if denom.item() == 0:
        return torch.tensor(0.0, device=preds.device, requires_grad=True)
    return diff.sum() / denom

In [73]:
batch = next(iter(train_loader))
input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
targets = batch["targets"].to(device)
valid_mask = batch["valid_mask"].to(device)

with torch.no_grad():
    preds = model(input_ids, attention_mask)

print("pred nan?", torch.isnan(preds).any().item())
print("loss:", masked_regression_loss(preds, targets, valid_mask).item())

pred nan? False
loss: 854.6677856445312


In [74]:
def train_one_epoch(model, loader, optimizer, device):
    model.train()
    total = 0.0

    for batch in tqdm(loader, leave=False):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].to(device)
        valid_mask = batch["valid_mask"].to(device)

        optimizer.zero_grad()
        preds = model(input_ids, attention_mask)
        loss = masked_regression_loss(preds, targets, valid_mask)
        loss.backward()
        optimizer.step()
        total += loss.item()

    return total / max(len(loader), 1)

def eval_one_epoch(model, loader, device):
    model.eval()
    total = 0.0

    with torch.no_grad():
        for batch in tqdm(loader, leave=False):
            input_ids = batch["input_ids"].to(device)
            attention_mask = batch["attention_mask"].to(device)
            targets = batch["targets"].to(device)
            valid_mask = batch["valid_mask"].to(device)

            preds = model(input_ids, attention_mask)
            loss = masked_regression_loss(preds, targets, valid_mask)
            total += loss.item()

    return total / max(len(loader), 1)

In [75]:
best_val = float("inf")
best_path = "best_protbert_regressor.pt"

for epoch in range(3):
    tr = train_one_epoch(model, train_loader, optimizer, device)
    va = eval_one_epoch(model, val_loader, device)
    print(f"Epoch {epoch+1}: train={tr:.6f}, val={va:.6f}")

    if va < best_val:
        best_val = va
        torch.save(model.state_dict(), best_path)

  0%|          | 0/16821 [00:00<?, ?it/s]

  0%|          | 0/223 [00:00<?, ?it/s]

Epoch 1: train=163.022816, val=130.155845


  0%|          | 0/16821 [00:00<?, ?it/s]

  0%|          | 0/223 [00:00<?, ?it/s]

Epoch 2: train=142.889551, val=124.889007


  0%|          | 0/16821 [00:00<?, ?it/s]

  0%|          | 0/223 [00:00<?, ?it/s]

Epoch 3: train=139.374320, val=120.766130


In [76]:
model.load_state_dict(torch.load(best_path, map_location=device))
model.eval()

all_true, all_pred = [], []

with torch.no_grad():
    for batch in tqdm(test_loader):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        targets = batch["targets"].cpu().numpy()
        valid_mask = batch["valid_mask"].cpu().numpy()

        preds = model(input_ids, attention_mask).cpu().numpy()

        for b in range(preds.shape[0]):
            m = valid_mask[b].astype(bool)
            all_true.append(targets[b][m])
            all_pred.append(preds[b][m])

all_true = np.vstack(all_true)
all_pred = np.vstack(all_pred)

rmse = np.sqrt(mean_squared_error(all_true, all_pred))
mae = mean_absolute_error(all_true, all_pred)
r2 = r2_score(all_true, all_pred)

print("RMSE:", rmse)
print("MAE:", mae)
print("R2:", r2)

  0%|          | 0/116 [00:00<?, ?it/s]

RMSE: 228.02684532199274
MAE: 125.67106628417969
R2: -0.6063099503517151
